# Notebook 02 — Parameter Estimation

**Member:** Nasya Putri Salsabila — Estimation Analyst (Role B)  
**Repository audited:** `pandas-dev/pandas`  

**Research questions addressed:**
- RQ1: What is the estimated probability that a pull request gets merged in pandas-dev/pandas?
- RQ2: Has the average issue closing duration changed significantly after a major pandas release?

**Depends on:** `data/clean/dataset.csv` — output dari `01_eda.ipynb` (Role A)  
**Feeds into:** `03_confidence_interval.ipynb` (Role C) dan `04_hypothesis_testing.ipynb` (Role D)

## AI Usage Disclosure

**Member:** Nasya Putri Salsabila — Estimation Analyst | **Tools used:** Claude

| Task | Tool | Prompt summary | Output modified? |
|------|------|----------------|------------------|
| Scaffold `estimator.py` dan struktur notebook | Claude | "Scaffold MLE functions per Tsun 2020 for pandas-dev/pandas audit" | Ya — verifikasi semua formula, sesuaikan kolom dengan output Role A |

**Written entirely without AI:**
- Derivasi MLE Bernoulli (Section 1.1)
- Derivasi MLE Poisson (Section 2.1)
- Semua markdown interpretation cells

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import sys
import json
sys.path.insert(0, '..')   # agar src/ bisa dibaca dari folder notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import beta as beta_dist

from src.estimator import (
    mle_bernoulli,
    mle_poisson,
    beta_posterior,
    log_likelihood_bernoulli,
    log_likelihood_poisson,
)

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')
print('Setup OK — semua imports berhasil.')

In [ ]:
# ── Load Dataset dari Role A ────────────────────────────────────────────────
df = pd.read_csv('../data/clean/dataset.csv',
                 parse_dates=['created_at', 'closed_at'])

print(f'Total records  : {len(df):,}')
print(f'Kolom dataset  : {list(df.columns)}')
print(f'\nContoh data:')
df.head(3)

In [ ]:
# ── Siapkan variabel untuk analisis ────────────────────────────────────────
# Bernoulli: is_closed (1=closed, 0=open) — dari kolom Role A
bernoulli_data = df['is_closed'].dropna().astype(int).values

# Poisson: jumlah comments per issue — dari kolom Role A
poisson_data = df['comments'].dropna().astype(float).values

print(f'Data Bernoulli (is_closed) : {len(bernoulli_data):,} observasi')
print(f'  Closed (1) : {bernoulli_data.sum():,}')
print(f'  Open   (0) : {(bernoulli_data == 0).sum():,}')
print(f'\nData Poisson (comments)    : {len(poisson_data):,} observasi')
print(f'  Mean       : {poisson_data.mean():.2f}')
print(f'  Max        : {poisson_data.max():.0f}')

---
## 1. Model Bernoulli — Probabilitas Issue Ditutup

### Latar Belakang

Setiap issue di `pandas-dev/pandas` memiliki dua kemungkinan status:
- **X = 1** → issue closed (ditutup)
- **X = 0** → issue masih open

Karena setiap issue adalah percobaan independen dengan dua hasil, kita modelkan dengan **Bernoulli(θ)**, di mana θ adalah probabilitas sebuah issue ditutup.

### 1.1 Derivasi MLE Bernoulli

**Likelihood function:**
$$L(\theta) = \prod_{i=1}^{n} \theta^{x_i}(1-\theta)^{1-x_i} = \theta^k (1-\theta)^{n-k}$$

**Log-likelihood:**
$$\ln L(\theta) = \text{[-]}$$

**Turunkan terhadap θ, samakan dengan nol** $\frac{d \ln L}{d\theta} = 0$:
$$\text{[-]}$$

**Solusi MLE:**
$$\hat{\theta}_{MLE} = \text{[-]}$$

In [ ]:
# ── 1.2 Hitung MLE Bernoulli ────────────────────────────────────────────────
result_bern = mle_bernoulli(bernoulli_data)

print('=== MLE Bernoulli — Issue Close Rate (pandas-dev/pandas) ===')
print(f"  k (closed issues) : {result_bern['k']:,}")
print(f"  n (total issues)  : {result_bern['n']:,}")
print(f"  θ̂  (MLE)          : {result_bern['theta_hat']:.4f}")
print(f"  SE(θ̂)             : {result_bern['se']:.4f}")

In [ ]:
# ── 1.3 Visualisasi Log-Likelihood Bernoulli ────────────────────────────────
k = result_bern['k']
n = result_bern['n']

theta_range = np.linspace(0.01, 0.99, 500)
ll_values   = [log_likelihood_bernoulli(t, k, n) for t in theta_range]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(theta_range, ll_values, color='steelblue', lw=2, label='ln L(θ)')
ax.axvline(result_bern['theta_hat'], color='tomato', ls='--', lw=1.5,
           label=f"θ̂ = {result_bern['theta_hat']:.4f}")
ax.set_xlabel('θ (probabilitas issue ditutup)')
ax.set_ylabel('Log-Likelihood  ln L(θ)')
ax.set_title('Bernoulli Log-Likelihood — pandas-dev/pandas Issue Close Rate')
ax.legend()
plt.tight_layout()
plt.savefig('../report/fig_ll_bernoulli.png', dpi=150)
plt.show()

### 1.4 Interpretasi MLE Bernoulli

> Jelaskan apa arti nilai θ̂ dalam konteks `pandas-dev/pandas`.  
> Apakah angka ini masuk akal?  
> Apa implikasinya bagi maintainer proyek?

*(tulis interpretasimu di sini)*

---
## 2. Model Poisson — Rata-rata Komentar per Issue

### Latar Belakang

Jumlah komentar per issue adalah **count data** (non-negatif, diskrit).  
Ini cocok dimodelkan dengan **Poisson(λ)**, di mana λ adalah rata-rata komentar per issue.

### 2.1 Derivasi MLE Poisson
**Likelihood function:**
$$L(\lambda) = \prod_{i=1}^{n} \frac{e^{-\lambda} \lambda^{x_i}}{x_i!}$$

**Log-likelihood:**
$$\ln L(\lambda) = \text{[-]}$$

**Turunkan terhadap λ, samakan dengan nol** $\frac{d \ln L}{d\lambda} = 0$:
$$\text{[-]}$$

**Solusi MLE:**
$$\hat{\lambda}_{MLE} = \text{[-]}$$

In [ ]:
# ── 2.2 Hitung MLE Poisson ──────────────────────────────────────────────────
result_pois = mle_poisson(poisson_data)

print('=== MLE Poisson — Comments per Issue (pandas-dev/pandas) ===')
print(f"  n (issues)     : {result_pois['n']:,}")
print(f"  λ̂  (MLE)       : {result_pois['lambda_hat']:.4f}")
print(f"  SE(λ̂)          : {result_pois['se']:.4f}")

In [ ]:
# ── 2.3 Visualisasi Log-Likelihood Poisson ─────────────────────────────────
lam_hat   = result_pois['lambda_hat']
lam_range = np.linspace(max(0.1, lam_hat - lam_hat*0.5),
                        lam_hat + lam_hat*0.5, 300)
ll_pois   = [log_likelihood_poisson(l, poisson_data) for l in lam_range]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lam_range, ll_pois, color='seagreen', lw=2, label='ln L(λ)')
ax.axvline(lam_hat, color='tomato', ls='--', lw=1.5,
           label=f"λ̂ = {lam_hat:.2f}")
ax.set_xlabel('λ (rata-rata komentar per issue)')
ax.set_ylabel('Log-Likelihood  ln L(λ)')
ax.set_title('Poisson Log-Likelihood — pandas-dev/pandas Comments per Issue')
ax.legend()
plt.tight_layout()
plt.savefig('../report/fig_ll_poisson.png', dpi=150)
plt.show()

### 2.4 Interpretasi MLE Poisson

> Jelaskan arti λ̂ dalam konteks `pandas-dev/pandas`.  
> Apakah rata-rata komentar ini mencerminkan tingkat diskusi yang aktif?  
> Apa yang bisa disimpulkan tentang engagement komunitas pandas?

*(tulis interpretasimu di sini)*

---
## 3. Beta Posterior — Bayesian Update untuk Issue Close Rate

In [ ]:
# ── 3.1 Hitung Beta Posterior ───────────────────────────────────────────────
k = result_bern['k']               # jumlah closed
m = result_bern['n'] - result_bern['k']   # jumlah open

result_beta = beta_posterior(k=k, m=m)

print('=== Beta Posterior — Issue Close Rate (pandas-dev/pandas) ===')
print(f"  k (closed)       : {k:,}")
print(f"  m (open)         : {m:,}")
print(f"  α = k+1          : {result_beta['alpha']:,}")
print(f"  β = m+1          : {result_beta['beta']:,}")
print(f"  Posterior mode   : {result_beta['mode']:.4f}")
print(f"  Posterior mean   : {result_beta['mean']:.4f}")
print(f"  95% credible int.: {result_beta['ci_95']}")

In [ ]:
# ── 3.2 Visualisasi Beta Posterior ─────────────────────────────────────────
alpha = result_beta['alpha']
beta  = result_beta['beta']
mean  = result_beta['mean']

lo = max(0.001, mean - 0.1)
hi = min(0.999, mean + 0.1)
theta_range = np.linspace(lo, hi, 500)
pdf_values  = beta_dist.pdf(theta_range, alpha, beta)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(theta_range, pdf_values, color='darkorchid', lw=2,
        label=f'Beta(α={alpha}, β={beta})')
ax.axvline(result_beta['mode'], color='tomato', ls='--', lw=1.5,
           label=f"Mode = {result_beta['mode']:.4f}")
ax.axvline(result_beta['ci_95'][0], color='gray', ls=':', lw=1)
ax.axvline(result_beta['ci_95'][1], color='gray', ls=':', lw=1,
           label=f"95% CI: {result_beta['ci_95']}")
ax.set_xlabel('θ (probabilitas issue ditutup)')
ax.set_ylabel('Density')
ax.set_title('Beta Posterior — Issue Close Rate (pandas-dev/pandas)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../report/fig_beta_posterior.png', dpi=150)
plt.show()

### 3.3 Interpretasi Beta Posterior

> Jelaskan perbedaan antara MLE θ̂ dan posterior mode.  
> Apa makna credible interval 95% ini?  
> Mengapa kita gunakan prior Beta(1,1)?  
> **Hati-hati:** credible interval ≠ confidence interval — jelaskan perbedaannya.

*(tulis interpretasimu di sini)*

---
## 4. Ringkasan & Handoff ke Layer Berikutnya

> Rangkum semua hasil estimasi untuk Role C dan D.

*(tulis ringkasanmu di sini)*

In [ ]:
# ── Ekspor hasil estimasi untuk Role C & D ────────────────────────────────
estimation_results = {
    'bernoulli': result_bern,
    'poisson'  : result_pois,
    'beta'     : {
        'alpha': result_beta['alpha'],
        'beta' : result_beta['beta'],
        'mode' : result_beta['mode'],
        'mean' : result_beta['mean'],
        'ci_95': list(result_beta['ci_95']),
    },
}

import os
os.makedirs('../data/clean', exist_ok=True)
with open('../data/clean/estimation_results.json', 'w') as f:
    json.dump(estimation_results, f, indent=2)

print('estimation_results.json tersimpan di data/clean/')
print(json.dumps(estimation_results, indent=2))